# 05 — Validazione del Caso C

Questo notebook raccoglie la validazione statistica della ricostruzione del Caso C:

* direzione comune $\Omega_n$;
* spettro gaussiano $E_n^{(k)}\sim N(\mu_E,\sigma_E)$.

La checklist si legge in questo ordine: **bias → risoluzione → pull → coverage → contrazione →
robustezza al prior**. Ogni passo ha senso solo se i precedenti sono a posto.

Il notebook non lancia run nuovi: legge i risultati salvati in `outputs/`.

* `caso_C_checklist_results.pkl`: la checklist principale. Si rigenera con
  `OMP_NUM_THREADS=1 python scripts/caso_C_checklist.py 4`, che richiede circa un'ora;
* `caso_C_cov68_results.pkl`, `caso_C_cov90_mu_results.pkl`: ripetizioni su seed nuovi per i punti
  dove la checklist era al limite;
* `caso_C_diag_omega_N50*.pkl`: diagnostica della coverage di $\Omega_n$ a N = 50;
* `n1000_extra_results.pkl`: 200 esperimenti aggiunti a N = 1000 (sezione 8);
* `robustezza_prior_results.pkl`, `stress_spettro*_results.pkl`, `stress_sorgente_results.pkl`,
  `correlazione_stadi_results.pkl`, `modo_log_sigma_results.pkl`: stress test delle
  assunzioni e del metodo a due stadi (sezioni 6, 9–11). Ciascuno si rigenera con lo script
  omonimo in `scripts/`.

Ogni esperimento estrae una verità nuova:

* $\Omega_n$ isotropa;
* $\mu_E\sim U(2.5, 4)$ MeV;
* $\sigma_E$ log-uniforme fra 0.2 e 0.6 MeV.

È una regione interna al prior della ricostruzione, che è più largo. La coverage non è quindi
garantita esatta per costruzione, come nel notebook 02, e va misurata. Le stime sono:

* $\mu_E$: media posteriore;
* $\sigma_E$: mediana posteriore di $\log\sigma_E$;
* $\Omega_n$: MAP sulla calotta.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

from riptide_toy.constants import SEED
from riptide_toy import validate

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(SEED)
OUT = Path("..") / "outputs"


def load(name: str):
    """Carica un pkl da outputs/."""
    with open(OUT / name, "rb") as f:
        return pickle.load(f)


def collect(results: list, n_events: int, prior: str = "logU") -> dict:
    """Raccoglie in array gli esperimenti della checklist a N fissato (stessa logica dello script)."""
    rows = [r for r in results if r["N"] == n_events]
    out = {k: np.array([r[k] for r in rows]) for k in
           ("mu_true", "sigma_true", "omega_error", "omega_sigma", "omega_covered")}
    for k in rows[0][prior]:
        out[k] = np.array([r[prior][k] for r in rows])
    return out


def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(x ** 2)))


def binom_err(p, m: int):
    """Errore binomiale di una coverage p stimata su m esperimenti."""
    return np.sqrt(np.asarray(p) * (1 - np.asarray(p)) / m)


CHECK = load("caso_C_checklist_results.pkl")
LEVELS = CHECK["LEVELS"]
N_VALUES = sorted(n for n, _ in CHECK["N_M"])
TABLE = {n: collect(CHECK["results"], n) for n in N_VALUES}
print("N, M:", CHECK["N_M"], "| livelli:", LEVELS)

## Esempio guidato: quanto è precisa una coverage, e cosa vuol dire pull per una direzione

**Errore binomiale.** Ogni esperimento dà un sì/no: la verità sta o non sta nell'intervallo. Con M
esperimenti e probabilità vera $p$, la frazione osservata ha deviazione standard
$\sqrt{p(1-p)/M}$ **(a)**.

* $M = 200$, $p = 0.68$: $\sqrt{0.68 \times 0.32/200} = \sqrt{0.001088} = 0.033$. Una coverage fra
  0.65 e 0.71 è quindi compatibile con il nominale;
* $M = 40$ (N = 1000): $\sqrt{0.2176/40} = 0.074$. Un valore osservato di 0.55 dista
  $(0.55 - 0.68)/0.074 = -1.8$ errori: è una fluttuazione plausibile, non una prova di difetto;
* per vedere uno scarto di 0.02 a $2\sigma$ servono $M = 0.2176/(0.01)^2 \approx 2200$ esperimenti.

**Pull per una direzione.** L'errore angolare $d$ è una distanza sulla sfera, quindi è sempre
$\ge 0$: non ha senso chiedere che la sua media sia 0. Se il posterior è circa gaussiano con
larghezza $s$ per asse, le due componenti dell'errore sono $N(0, s)$ e $d^2 = s^2\chi^2_2$, con
$E[d^2] = 2s^2$. Il pull $d/\sqrt{E_\text{post}[d^2]}$ ha quindi **rms 1** e $\sqrt2\cdot$pull segue una
Rayleigh **(a)**. La cella conferma entrambe le affermazioni.

In [ ]:
for m in (200, 40):
    print(f"M={m:3d}: errore binomiale a 0.68 = {binom_err(0.68, m):.3f}")
print(f"0.55 a M=40: z = {(0.55 - 0.68) / binom_err(0.68, 40):+.2f}")

s = 0.02  # rad, larghezza per asse
d = np.hypot(*rng.normal(0, s, (2, 100_000)))
pull = d / np.sqrt(2 * s**2)
print(f"pull angolare simulato: rms {rms(pull):.3f} (atteso 1); "
      f"mediana di sqrt(2)*pull {np.median(np.sqrt(2) * pull):.3f} (Rayleigh: sqrt(2 ln 2) = {np.sqrt(2 * np.log(2)):.3f})")

## 1–2. Bias e risoluzione

Per ogni N la tabella dà:

* il bias medio, con il suo errore $\text{sd}/\sqrt M$;
* l'rms dell'errore reale, confrontato con la larghezza media dichiarata dal posterior (fra
  parentesi).

Il bias di $\log\sigma_E$ si misura sulla mediana posteriore.

In [ ]:
for n in N_VALUES:
    t = TABLE[n]
    m = t["mu_true"].size
    mu_res = t["mu_mean"] - t["mu_true"]
    ls_res = t["ls_median"] - np.log(t["sigma_true"])
    print(f"N={n:4d} M={m:3d} | bias mu {mu_res.mean():+.4f} +- {mu_res.std() / np.sqrt(m):.4f} MeV"
          f" | bias log sigma {ls_res.mean():+.3f} +- {ls_res.std() / np.sqrt(m):.3f}"
          f" | errore medio Omega {np.degrees(t['omega_error'].mean()):.2f} gradi")
    print(f"{'':11s} | rms mu {rms(mu_res):.4f} ({t['mu_std'].mean():.4f})"
          f" | rms log sigma {rms(ls_res):.3f} ({t['ls_std'].mean():.3f})"
          f" | rms Omega {np.degrees(rms(t['omega_error'])):.2f} ({np.degrees(t['omega_sigma'].mean()):.2f}) gradi")

Lettura **(c)**:

* **$\mu_E$**: bias compatibile con zero a ogni N (per esempio $+0.002 \pm 0.006$ MeV a N = 50) e
  risoluzione dichiarata uguale all'rms reale;
* **$\Omega_n$**: risoluzione dichiarata uguale all'rms reale entro pochi percento (1.87° contro 1.97°
  a N = 50);
* **$\log\sigma_E$**: a N = 50 il bias è $-0.15$, cioè la stima è trascinata verso i $\sigma_E$ piccoli,
  e si riduce con N ($-0.06$ a N = 150, $-0.02$ a N = 300). Non è un difetto del codice: con 50 eventi
  la larghezza dello spettro è poco determinata (la risoluzione su $E_n$ del singolo evento è
  confrontabile con $\sigma_E$) e la mediana posteriore riflette il prior in $\log\sigma_E$. Il
  posterior è però abbastanza largo da contenere lo spostamento: pull e coverage restano nominali
  (sezioni 3–4).

## 3. Pull

Per $\mu_E$ e $\log\sigma_E$ il pull è $(\text{media} - \text{vero})/\text{sd}$ e deve essere $N(0,1)$:
media 0 e larghezza 1. Per $\Omega_n$ conta l'rms, come nell'esempio guidato.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
x = np.linspace(-4, 4, 200)
for n in N_VALUES:
    t = TABLE[n]
    mu_pull = (t["mu_mean"] - t["mu_true"]) / t["mu_std"]
    ls_pull = (t["ls_mean"] - np.log(t["sigma_true"])) / t["ls_std"]
    om_pull = validate.angular_pull(t["omega_error"], t["omega_sigma"])
    print(f"N={n:4d} | pull mu {mu_pull.mean():+.2f} / {mu_pull.std():.2f}"
          f" | pull log sigma {ls_pull.mean():+.2f} / {ls_pull.std():.2f} | rms pull Omega {rms(om_pull):.2f}")
    if n == 150:
        for ax, p, name in zip(axes[:2], (mu_pull, ls_pull), ("mu", "log sigma")):
            ax.hist(p, bins=25, range=(-4, 4), density=True, alpha=0.7)
            ax.plot(x, norm.pdf(x), "k-"); ax.set_title(f"pull {name}, N=150")
        r = np.linspace(0, 4, 200)
        axes[2].hist(np.sqrt(2) * om_pull, bins=25, range=(0, 4), density=True, alpha=0.7)
        axes[2].plot(r, r * np.exp(-r**2 / 2), "k-"); axes[2].set_title("sqrt(2) pull Omega vs Rayleigh, N=150")
fig.tight_layout(); plt.show()

Lettura **(c)**:

* per N = 50–300 i pull hanno media vicina a 0 e larghezza vicina a 1 (fra 0.96 e 1.14, con M = 200
  l'errore sulla larghezza è circa $1/\sqrt{2M} = 0.05$);
* l'rms del pull angolare va da 1.00 a 1.05;
* a N = 1000 (M = 40) il pull di $\Omega_n$ ha rms 0.89 e quello di $\log\sigma_E$ larghezza 0.87.
  Con 40 esperimenti è al limite della significatività; con M = 240 (sezione 8) tornano a 1.01 e 0.94.

## 4. Coverage

Frazione di esperimenti in cui la verità sta nell'intervallo credibile al 68 / 90 / 95%:

* $\mu_E$ e $\log\sigma_E$: intervalli centrali;
* $\Omega_n$: regione HPD sulla sfera.

Fra parentesi è riportato l'errore binomiale al 68%.

In [ ]:
cov_table = {}
for n in N_VALUES:
    t = TABLE[n]
    m = t["mu_true"].size
    _, mu_cov = validate.coverage_curve(t["mu_true"], t["mu_int"], LEVELS)
    _, ls_cov = validate.coverage_curve(np.log(t["sigma_true"]), t["ls_int"], LEVELS)
    om_cov = t["omega_covered"].mean(axis=0)
    cov_table[n] = (mu_cov, ls_cov, om_cov, m)
    print(f"N={n:4d} (+-{binom_err(0.68, m):.3f}) | mu {np.round(mu_cov, 2)} | log sigma {np.round(ls_cov, 2)}"
          f" | Omega {np.round(om_cov, 2)}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, k, name in zip(axes, range(3), ("mu", "log sigma", "Omega")):
    for j, n in enumerate(N_VALUES):
        cov, m = cov_table[n][k], cov_table[n][3]
        ax.errorbar(LEVELS + 0.005 * (j - 1.5), cov - LEVELS, yerr=binom_err(LEVELS, m), fmt="o", capsize=3,
                    label=f"N={n}")
    ax.axhline(0, color="k", lw=0.8); ax.set(title=name, xlabel="livello nominale")
axes[0].set_ylabel("coverage - nominale"); axes[0].legend(fontsize=8)
fig.tight_layout(); plt.show()

Quasi tutti i punti sono entro 2 errori binomiali dal nominale **(c)**. Tre sono al limite e sono stati
ricontrollati con run dedicati su seed nuovi, nella sezione 7:

* $\log\sigma_E$ al 68% a N = 150, che vale 0.60;
* $\mu_E$ al 90% a N = 300, che vale 0.84;
* $\Omega_n$ a N = 50.

## 5. Contrazione

Se gli eventi sono indipendenti e il sistematico condiviso è trascurabile, l'rms dell'errore scala
come $1/\sqrt N$. Su scala log-log è una retta di pendenza $-0.5$, e il prodotto $\text{rms}\cdot\sqrt N$
è costante. Un plateau sarebbe la firma di un errore comune a tutti gli eventi.

In [ ]:
curves = {
    "mu [MeV]": [rms(TABLE[n]["mu_mean"] - TABLE[n]["mu_true"]) for n in N_VALUES],
    "log sigma": [rms(TABLE[n]["ls_median"] - np.log(TABLE[n]["sigma_true"])) for n in N_VALUES],
    "Omega [rad]": [rms(TABLE[n]["omega_error"]) for n in N_VALUES],
}
fig, ax = plt.subplots(figsize=(5.5, 4))
for name, curve in curves.items():
    slope = np.polyfit(np.log(N_VALUES), np.log(curve), 1)[0]
    print(f"{name:12s} pendenza {slope:+.3f}   rms*sqrt(N) = {np.round(np.array(curve) * np.sqrt(N_VALUES), 3)}")
    ax.loglog(N_VALUES, curve, "o-", label=name)
    ax.loglog(N_VALUES, curve[0] * np.sqrt(N_VALUES[0] / np.array(N_VALUES)), "k:", lw=0.8)
ax.set(xlabel="N eventi", ylabel="rms errore", title="contrazione (punteggiato: 1/sqrt(N))"); ax.legend()
plt.show()

Lettura **(c)**:

* **$\mu_E$**: pendenza $-0.46$, con $\text{rms}\cdot\sqrt N$ fra 0.57 e 0.69 MeV;
* **$\Omega_n$**: pendenza $-0.56$, con $\text{rms}\cdot\sqrt N$ da 0.24 a 0.20 rad. Per esempio a
  N = 300 ci si aspetta $\approx 0.21/\sqrt{300} = 0.012$ rad $\approx 0.7^\circ$;
* **$\log\sigma_E$**: pendenza $-0.74$, più ripida di $-0.5$. A N piccolo l'errore è gonfiato dal
  bias verso il prior (sezione 1–2), che si riduce con N, e quindi la curva scende più in fretta
  di $1/\sqrt N$.

In nessun caso compare un plateau fino a N = 1000.

## 6. Robustezza al prior

Lo stadio 2 è ripetuto sugli stessi dati cambiando prior: uniforme in $\sigma_E$ invece che in
$\log\sigma_E$ (`scripts/robustezza_prior.py`, N = 50 / 150 / 300 / 1000). Lo spostamento delle
stime si misura in unità di deviazione standard posteriore del default.

In [ ]:
import sys

sys.path.insert(0, "../scripts")
import robustezza_prior

ROB = load("robustezza_prior_results.pkl")
for key, title in (("mu", "Caso C, mu_E (MeV)"), ("ls", "Caso C, log sigma_E")):
    robustezza_prior.print_scalar_table(title, ROB["results"]["C"], ["logU", "U"], key, f"{key}_true")

Lettura **(c)**:

* $\mu_E$ è insensibile al prior: lo spostamento medio è fra $-0.02$ e $0.00$ deviazioni standard a
  ogni N;
* $\log\sigma_E$ si sposta in media di $+0.32 / +0.17 / +0.11 / +0.06$ deviazioni standard per
  N = 50 / 150 / 300 / 1000. Il prior uniforme in $\sigma_E$ favorisce gli spettri larghi, e il suo
  peso scende con N come per ogni parametro condiviso;
* la coverage di $\log\sigma_E$ col prior uniforme sta fra 0.59 e 0.70 al 68%, uguale a quella
  del default entro gli errori: il prior sposta la stima ma non rompe la calibrazione.

Fino a N ≈ 150 la larghezza dello spettro dipende quindi ancora in parte dal prior, e va
dichiarata insieme al risultato.

## 7. Ricontrolli dei punti al limite

**$\log\sigma_E$ al 68% e $\mu_E$ al 90%.** Si ripete lo stesso esperimento su seed nuovi:

* `cov68`: N = 150 con M = 600 e N = 300 con M = 300;
* `cov90`: N = 300 con M = 600.

Se i valori della checklist erano fluttuazioni, i nuovi run devono tornare al nominale entro il
loro errore binomiale, che è più piccolo.

In [ ]:
COV68, COV90 = load("caso_C_cov68_results.pkl"), load("caso_C_cov90_mu_results.pkl")


def rows_of(pkl: dict, n: int, key: str, which: str = "hat") -> np.ndarray:
    return np.array([r[which][key] if key not in ("mu_true", "sigma_true") else r[key]
                     for r in pkl["results"] if r["N"] == n])


pooled = []
for n in (150, 300):
    truth, ints = np.log(rows_of(COV68, n, "sigma_true")), rows_of(COV68, n, "ls_int")
    _, cov = validate.coverage_curve(truth, ints, LEVELS)
    pull = (rows_of(COV68, n, "ls_mean") - truth) / rows_of(COV68, n, "ls_std")
    pooled.append((cov[0], truth.size))
    print(f"cov68 N={n}: log sigma coverage 68% = {cov[0]:.3f} +- {binom_err(cov[0], truth.size):.3f}"
          f"  (M={truth.size}, pull sd {pull.std():.2f})")
m = sum(k for _, k in pooled)
print(f"combinato M={m}: {sum(c * k for c, k in pooled) / m:.3f} +- {binom_err(0.68, m):.3f}")

truth, ints = rows_of(COV90, 300, "mu_true"), rows_of(COV90, 300, "mu_int")
_, cov = validate.coverage_curve(truth, ints, LEVELS)
print(f"cov90 N=300: mu coverage {np.round(cov, 3)}, al 90% {cov[1]:.3f} +- {binom_err(cov[1], truth.size):.3f}"
      f" (M={truth.size})")

Lettura **(c)**:

* **$\log\sigma_E$ al 68%**: 0.658 a N = 150 e 0.723 a N = 300, entrambi entro 1.7 errori
  binomiali. Combinando i 900 esperimenti si ottiene 0.68: lo 0.60 della checklist era una
  fluttuazione;
* **$\mu_E$ al 90% a N = 300**: è nominale sui 600 seed nuovi, quindi lo 0.84 della checklist era
  una fluttuazione.

**$\Omega_n$ a N = 50.** Lo stadio 1 stima la direzione con un prior sullo spettro, e la scelta di
quel prior conta quando gli eventi sono pochi. Si sono confrontate quattro varianti sugli stessi
dati:

* *plug-in*: la gaussiana con le stime puntuali di $(\mu_E, \sigma_E)$;
* *oracolo*: la gaussiana con i valori veri, non disponibile nei dati reali;
* *predittiva*: $N\big(E[\mu_E], \sqrt{E[\sigma_E^2] + \mathrm{Var}\,\mu_E}\big)$, che include
  l'incertezza sugli iperparametri;
* *plug-in +1 sd*: il plug-in con $\sigma_E$ allargata di una deviazione standard.

In [ ]:
DIAG = {"checklist": load("caso_C_diag_omega_N50.pkl"), "seed 50": load("caso_C_diag_omega_N50_tag50.pkl"),
        "seed 51": load("caso_C_diag_omega_N50_tag51.pkl")}
for v in ("plugin", "oracolo", "predittiva", "plugin_+1sd"):
    parts = []
    for pkl in DIAG.values():
        res = pkl["results"]
        if v in res[0]:
            parts.append((np.mean([r[v]["cov"] for r in res], axis=0), len(res),
                          validate.angular_pull(np.array([r[v]["err"] for r in res]), np.array([r[v]["sigma"] for r in res]))))
    m = sum(p[1] for p in parts)
    cov = sum(p[0] * p[1] for p in parts) / m
    pull = np.concatenate([p[2] for p in parts])
    z = (cov - LEVELS) / binom_err(LEVELS, m)
    print(f"[{v:11s}] M={m:4d} coverage {np.round(cov, 3)}  z {np.round(z, 1)}  pull rms {rms(pull):.2f}")

Lettura **(c)**:

* il plug-in sottocopre di circa 2 errori binomiali, perché tratta come esatte stime di
  $(\mu_E,\sigma_E)$ che con 50 eventi sono incerte;
* l'oracolo e la predittiva sono nominali;
* la predittiva è la variante adottata nello stadio 1 iterato, e su questa sono calcolati i
  numeri delle sezioni 1–6.

## 8. N = 1000 a statistica piena

La checklist ha solo M = 40 esperimenti a N = 1000, con un errore binomiale di ±0.074 al 68%.
`scripts/n1000_extra.py` rilancia la stessa pipeline su 200 seed disgiunti (indici 40–239), che si
uniscono a quelli della checklist: l'errore scende a ±0.030.

In [ ]:
import n1000_extra
import mu_N1000_diag

EXTRA = load("n1000_extra_results.pkl")
rows_1000 = [r for r in CHECK["results"] if r["N"] == 1000] + EXTRA["results"]["C"]
print("Omega_n:     ", n1000_extra.omega_summary(rows_1000))
print("iperparametri", n1000_extra.hyper_summary(rows_1000))
print()
mu_N1000_diag.main()

Lettura **(c)**:

* con M = 240 tutto è nominale: $\Omega_n$ rms 0.42° (dichiarata 0.41°), pull rms 1.01, coverage
  0.67 / 0.90 / 0.95; pull di $\mu_E$ +0.01 / 0.92 e di $\log\sigma_E$ −0.09 / 0.94; bias di
  $\mu_E$ $+0.0002\pm0.0012$ MeV. Lo 0.55 al 68% per $\mu_E$ a M = 40 era una fluttuazione;
* resta $\mu_E$ un po' sovra-coperto al 90% (0.95 ± 0.019), con std posteriore circa 8% più larga
  della dispersione. `scripts/mu_N1000_diag.py` lo valuta in blocco: larghezza del pull a −1.7
  $\sigma$, coverage al 90% / 95% a +2.6 / +2.4 $\sigma$ ma fortemente correlate. Con la covarianza
  di pseudo-esperimenti calibrati la p-value globale è 0.08. Non c'è struttura per terzili di
  $\sigma_E$ o $\mu_E$, e il passo della griglia fine non incide sui momenti. È compatibile con una
  fluttuazione.

## 9. Spettro non gaussiano

Lo stress test dell'assunzione 3 (notebook 04) usa gli stessi seed della checklist, quindi la
forma `gauss` riproduce esattamente i numeri delle sezioni 1–5. Qui sotto N = 150 e 300 per tutte
le forme; poi la verifica del modello di errore $\text{var}(\log s)\approx(\kappa-1)/(4N)$ sui
risultati salvati (`scripts/curtosi_log_sigma.py`).

In [ ]:
import stress_spettro
import curtosi_log_sigma

SPEC, SPEC_TAILS = load("stress_spettro_results.pkl"), load("stress_spettro_code_results.pkl")
rows_spec = [r for r in SPEC["results"] + SPEC_TAILS["results"] if r["N"] in (150, 300)]
stress_spettro.print_table(rows_spec, SPEC["SHAPES"] + SPEC_TAILS["SHAPES"])
print()
curtosi_log_sigma.main()

Lettura:

* **$\Omega_n$ e $\mu_E$** non risentono della forma: bias nullo, pull e coverage nominali per tutte
  le forme **(c)**. A N = 300 i pull medi di $\mu_E$ fino a +0.2 per uniforme e Laplace stanno entro
  2 errori (±0.1). Fa eccezione la Student-t, con pull medio di $\mu_E$ $+0.16 \pm 0.04$ combinando
  tutti gli N (qui +0.10 / +0.04 a N = 150 / 300; coverage e larghezza nominali); non viene dagli esperimenti con energie fuori da [0.5, 6] MeV.
  Causa non indagata **(d)**;
* **$\log\sigma_E$**: la larghezza del pull a N = 150 / 300 è 0.84 / 0.77 per l'uniforme, 0.80 /
  0.91 per la bimodale, 1.23 / 1.20 per la Laplace e 1.17 / 1.39 per la Student-t. La coverage 68%
  è 0.75–0.81 per curtosi bassa e 0.56–0.65 per code pesanti **(c)**;
* **previsione scritta prima del run delle code pesanti** (solo il termine intrinseco aggiunto alla
  larghezza gaussiana): Laplace 1.17 / 1.19, Student-t 1.28 / 1.30. È confermata entro gli errori
  (circa ±0.06–0.10) **(c)**. L'alternativa con il termine amplificato era 1.44–1.78, esclusa a più
  di 3 $\sigma$;
* a N = 1000 (M = 40) le larghezze osservate con code pesanti (1.41, 1.58) superano la previsione
  di 2.2–2.6 $\sigma$. Non indagato **(d)**.

In pratica: se lo spettro fisico può avere code, l'incertezza su $\sigma_E$ del modello gaussiano va
presa come limite inferiore.

## 10. Seconda sorgente

Come nel notebook 03, una frazione $f$ di eventi viene da una seconda sorgente a separazione
$\Delta$ o da un fondo isotropo. Nel Caso C anche gli iperparametri ne risentono: gli eventi da
un'altra direzione vengono ricostruiti con l'angolo sbagliato, quindi con energie disperse.

In [ ]:
SRC = load("stress_sorgente_results.pkl")


def source_table(case: str, n: int) -> None:
    sel_n = [r for r in SRC["results"] if r["case"] == case and r["N"] == n]
    thr = np.quantile([r["ks"] for r in sel_n if r["f"] == 0], 0.95)
    print(f"Caso {case}, N = {n}; soglia KS (95 percentile a f=0) = {thr:.4f}")
    configs = [(0.0, None)] + [(f, s) for s in SRC["SEPARATIONS_DEG"] for f in SRC["FRACTIONS"]]
    for frac, sep in configs:
        sel = [r for r in sel_n if r["f"] == frac and r["sep"] == sep]
        err = np.array([r["omega_error"] for r in sel]); sig = np.array([r["omega_sigma"] for r in sel])
        label = "nessuna" if frac == 0 else f"f={frac:.2f} " + ("isotropo" if sep is None else f"{sep:.0f} gradi")
        line = (f"  {label:20s} M={len(sel):3d} | Omega pull rms {rms(validate.angular_pull(err, sig)):5.2f}"
                f" cov68 {np.mean([r['omega_covered'][0] for r in sel]):.2f}"
                f" | potenza KS {np.mean([r['ks'] > thr for r in sel]):.2f}")
        for key in ("mu", "ls"):
            truth = np.array([r[f"{key}_true"] for r in sel])
            pull = np.array([(r[key]["mean"] - r[f"{key}_true"]) / r[key]["std"] for r in sel])
            cov = np.mean([(r[key]["int"][0, 0] <= t) & (t <= r[key]["int"][0, 1]) for r, t in zip(sel, truth)])
            line += f" | {key}: pull {pull.mean():+5.2f} cov68 {cov:.2f}"
        print(line)


for n in SRC["N_VALUES"]["C"]:
    source_table("C", n); print()

Lettura **(c)**, a N = 300:

* senza contaminazione si ritrovano i numeri della checklist (pull medi −0.05 / −0.07);
* **$\sigma_E$ è il parametro più sensibile**: con $f = 0.05$ a 30° il pull medio di $\log\sigma_E$ è
  +3.95 (+7.6 a $f = 0.1$), mentre $\Omega_n$ ha già coverage 68% 0.11;
* **$\mu_E$** si sposta verso l'alto: poco per $\Delta \le 30°$ e $f \le 0.1$ (pull ≤ +0.8), molto a
  90° o col fondo isotropo (pull +2.5 con $f = 0.05$ a 90°, +3.0 con $f = 0.1$ isotropo);
* il test KS ha la stessa cecità del Caso B per $\Delta \le 30°$.

Un $\sigma_E$ più largo del previsto è quindi un sintomo, ma è utile solo se si ha un'aspettativa
indipendente sulla larghezza dello spettro **(d)**.

## 11. Due stadi contro stima congiunta

I due stadi fissano $\Omega_n$ quando stimano $(\mu_E,\sigma_E)$, e quindi ignorano la loro
correlazione posteriore. `scripts/correlazione_stadi.py` la misura senza griglia 4D:
sugli stessi dataset della checklist (M = 100 per N), calcola l'approssimazione di Laplace del
posterior congiunto 4D attorno alle stime dei due stadi. Si confrontano:

* la correlazione canonica fra la direzione e gli iperparametri;
* l'allargamento delle incertezze passando da condizionate a marginali;
* il pull del modo congiunto.

`scripts/modo_log_sigma.py` scompone poi lo scarto fra il modo congiunto e la media dei due stadi
su $\log\sigma_E$.

In [ ]:
import correlazione_stadi
import modo_log_sigma

correlazione_stadi.print_table(load("correlazione_stadi_results.pkl")["results"])
print()
modo_log_sigma.print_table(load("modo_log_sigma_results.pkl")["results"])

Lettura:

* la correlazione canonica è debole e scende con N: mediana 0.17 / 0.10 / 0.08 a N = 50 / 150 /
  300. Ignorarla stringe le incertezze in media dell'1% a N = 50 e dello 0.2–0.3% a N = 300 (caso
  peggiore 6%). **La fattorizzazione a due stadi è giustificata (c)**;
* le larghezze di Laplace concordano con le griglie dei due stadi entro l'1–7% **(c)**;
* il modo congiunto ha un pull medio negativo su $\log\sigma_E$ (−0.27 → −0.13), mentre la media
  dei due stadi è centrata. Non è l'asimmetria del marginale, che spinge nel verso opposto (+0.16 /
  +0.11 / +0.06 s). Il termine dominante viene dal lasciare libera $\Omega_n$ nel massimo
  congiunto: −0.28 / −0.20 / −0.14 s **(c)**. Tutti i termini scalano come $s^2$, come un effetto di
  volume: il modo di una densità a più dimensioni non conta il volume delle direzioni
  massimizzate (interpretazione, **d**);
* per le stime puntuali la media dei due stadi è migliore del modo congiunto **(c)**.

## Scelte di implementazione che hanno portato qui

Ognuna è nata da un difetto visto nella checklist e corretto. I dettagli sono in
`docs/report_caso_C_stadio1.md`.

* **Kernel sferico (von Mises–Fisher) per la risoluzione della traccia.** Con un kernel gaussiano
  piatto in $\theta$, $\mu_E$ aveva un bias di circa $-\sigma_\theta^2\mu_E \approx -0.02$ MeV, che scalava
  con $\sigma_\theta^2$. Con il kernel sferico il bias è compatibile con zero.
* **Calotta attorno alla MAP.** Il raggio della calotta su cui si raffina $\Omega_n$ era degenerato
  ($\approx 10^{-8}$ rad) quando la MAP veniva confrontata con se stessa, e dava pull di $10^5$. Ora la
  MAP si esclude per indice.
* **Stadio 1 iterato con prior predittivo.** Il pull di $\Omega_n$ è passato da rms 0.78 a 1.05
  (sezione 7).
* **Stadio 2 sulla direzione $\hat\Omega_0$ a prior largo.** Usare la direzione iterata peggiorava il
  bias di $\log\sigma_E$ a N = 50 ($-0.27$ contro $-0.15$).
* **Mai una griglia 4D bruta.** Il costo sarebbe $n_\text{pixel}\cdot n_\mu\cdot n_\sigma\cdot n_E$. Si usano
  invece due stadi: prima la direzione, poi una griglia 2D su $(\mu_E, \sigma_E)$.

## Limiti dichiarati

* **Due stadi**: $\hat\Omega_n$ e $(\hat\mu_E, \hat\sigma_E)$ vengono da stadi diversi. La correlazione
  trascurata costa ≤ 1% sulle incertezze in media (≤ 6% nel caso peggiore) a N ≥ 50: chiuso come
  limite **(c)**, sezione 11.
* **Assunzione 1 (sorgente unica)**: necessaria e non verificabile per sorgenti vicine. Con il 5%
  di eventi da una seconda sorgente a 30°, la coverage 68% di $\Omega_n$ scende a 0.11 e il pull di
  $\log\sigma_E$ arriva a +4 a N = 300; il test KS non la vede **(c)**, sezione 10.
* **Assunzione 3 (spettro gaussiano)**: $\mu_E$ e $\Omega_n$ sono robusti alla forma; la coverage di
  $\log\sigma_E$ dipende dalla curtosi: sovra-coperta per $\kappa < 3$, sotto-coperta con code
  pesanti (coverage 68% circa 0.6) **(c)**, sezione 9. Con la Student-t $\mu_E$ ha un pull medio di
  +0.16 **(c, causa d)**; a N = 1000 le larghezze del pull di $\log\sigma_E$ con code pesanti
  superano la previsione **(d)**.
* La larghezza dello spettro a N ≲ 150 dipende in parte dal prior (sezione 6).
* Isotropia nel CM assunta nel range 0.5–6 MeV **(b)**.